# Tool use with Claude

## Multi-turn conversations with tools

In [9]:
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import ToolParam, Message
from datetime import datetime, timedelta
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def get_current_datetime(date_format = "%Y-%m-%d %H:%M:%S"):

    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Gets the current system date and time, formatted according to a strftime-style format pattern. Use this whenever you need to know the present date or time (for example, to record when something happens, compute deadlines relative to 'today', or include a timestamp in a response). Do not guess the current date: call this tool to obtain it. Returns a single string containing the already-formatted date/time.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A Python strftime-style format string that defines how the returned date/time is represented. Must be a non-empty string containing at least one valid format code. Common codes: %Y (4-digit year, e.g. 2026), %m (month 01-12), %d (day 01-31), %H (hour 00-23), %M (minutes 00-59), %S (seconds 00-59). Examples: '%Y-%m-%d %H:%M:%S' -> '2026-08-29 14:30:00'; '%d/%m/%Y' -> '29/08/2026'; '%H:%M' -> '14:30'. If omitted, the default '%Y-%m-%d %H:%M:%S' is used.",
                    "default": "%Y-%m-%d %H:%M:%S"
                }
            },
            "required": []
        }
    }
)

In [4]:
def add_user_message(messages, message):
    user_message = {
        "role": "user"
        , "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant"
        , "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences = []):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
messages = []

add_user_message(messages, "Whats the current time in HH:MM:SS format?")

response = client.messages.create(
    model = model
    , max_tokens = 1000
    , messages = messages
    , tools = [get_current_datetime_schema]
)

print(response)

Message(id='msg_011CeXjzCEAzQSxiw7UYEN7d', container=None, content=[ToolUseBlock(id='toolu_012BerYFqPPW8E65kWgkCgBD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=849, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [6]:
add_assistant_message(messages, response)
print(messages)

[{'role': 'user', 'content': 'Whats the current time in HH:MM:SS format?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_012BerYFqPPW8E65kWgkCgBD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


In [30]:
def chat(messages, system = None, temperature = 1.0, stop_sequences = [], tools = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return "\n".join([block.text])
        else:
            return ""
    #return "\n".join([block.text for block in message.content if block.type == "text"])

## Implementing multiple turns

In [8]:
messages = []

add_user_message(messages, "Whats the current time in HH:MM:SS format?")

response = client.messages.create(
    model = model
    , max_tokens = 1000
    , messages = messages
    , tools = [get_current_datetime_schema]
)

print(response)

Message(id='msg_011CeXmYUHacRNNFQDAkry2n', container=None, content=[ToolUseBlock(id='toolu_017mZPsv5MLruWngcTXiTeQc', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=849, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [19]:
def run_tool(tool_name, tool_input):
    match tool_name:
        case "get_current_datetime":
            return get_current_datetime(**tool_input)
        case _:
            raise ValueError("No function available that matches this name")

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result"
                , "tool_use_id": tool_request.id
                , "content": json.dumps(tool_output)
                , "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result"
                , "tool_use_id": tool_request.id
                , "content": f"Error: {e}"
                , "is_error": True
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [32]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools = [get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response) if text_from_message(response) != "" else "Calling tool...")

        if response.stop_reason != "tool_use":
            break

        tool_results= run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [33]:
messages = []

add_user_message(messages, "What is the current time in HH:MM format? Also, what is the current time in SS format?")

run_conversation(messages)

Calling tool...
The current time is:
- **HH:MM format**: 16:50
- **SS format (seconds)**: 33


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01SzR5533uZtvMrD888QUydH', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01UZfTRgBdPh7MfBNjpgGAyf', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01SzR5533uZtvMrD888QUydH',
    'content': '"16:50"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01UZfTRgBdPh7MfBNjpgGAyf',
    'content': '"33"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 16:50\n- **SS format (seconds)**: 33', type='text')]}]